# Create NAFOSTED (Vietnam) Awards

**NAFOSTED** — National Foundation for Science & Technology Development (funder_id `4320309617`,
ROR `04rw64z44`, VN, priority `220`). Source: the public no-login project lookup at
dichvucong.nafosted.gov.vn/tra-cuu-de-tai.php (page stepped by 30; see scripts/local/nafosted_to_s3.py).
**7,071 grants**, native `Mã đề tài` ids (e.g. KC.09.14/21-30), **100% title / 99.8% person PI
(Chủ nhiệm đề tài, title-stripped) / 99.6% institution / 97.4% start+end dates (dd/MM/yyyy)**;
**no amount** (not published — §6.7 waiver). Vietnam's flagship competitive basic-research funder.
Existing OpenAlex coverage was Crossref-only.

Parquet: `s3://openalex-ingest/awards/nafosted/nafosted_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nafosted_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nafosted/nafosted_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.nafosted_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.nafosted_raw LIMIT 5;

## Step 2: Create NAFOSTED Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nafosted_awards
USING delta
AS
WITH
fndr AS (
    SELECT CAST(funder_id AS BIGINT) as funder_id, display_name, ror_id, doi
    FROM openalex.common.funder WHERE funder_id = 4320309617
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title), ''), CONCAT('NAFOSTED grant ', g.funder_award_id)) as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DECIMAL(18,2)) > 0 THEN TRY_CAST(g.amount AS DECIMAL(18,2)) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DECIMAL(18,2)) > 0 THEN g.currency ELSE NULL END as currency,
        struct(CONCAT('https://openalex.org/F', f.funder_id) as id, f.display_name, f.ror_id, f.doi) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'nafosted' as provenance,
        TRY_TO_DATE(g.start_date_raw, 'dd/MM/yyyy') as start_date,
        TRY_TO_DATE(g.end_date_raw, 'dd/MM/yyyy') as end_date,
        YEAR(TRY_TO_DATE(g.start_date_raw, 'dd/MM/yyyy')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date_raw, 'dd/MM/yyyy')) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(g.pi_given as given_name, g.pi_family as family_name, CAST(NULL AS STRING) as orcid, CAST(NULL AS DATE) as role_start,
                    struct(g.institution as name, 'Vietnam' as country, CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids) as affiliation)
            WHEN g.institution IS NOT NULL THEN
                struct(CAST(NULL AS STRING) as given_name, CAST(NULL AS STRING) as family_name, CAST(NULL AS STRING) as orcid, CAST(NULL AS DATE) as role_start,
                    struct(g.institution as name, 'Vietnam' as country, CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids) as affiliation)
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.nafosted_raw g
    CROSS JOIN fndr f
    WHERE g.funder_award_id IS NOT NULL
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw WHERE provenance = 'nafosted' AND priority = 220;
INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id, amount, currency, funder, funding_type, funder_scheme, provenance, start_date, end_date, start_year, end_year, lead_investigator, co_lead_investigator, investigators, landing_page_url, doi, works_api_url, created_date, updated_date, 220 as priority
FROM openalex.awards.nafosted_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(DISTINCT funder_award_id) uniq_award, COUNT(DISTINCT id) uniq_id, COUNT(DISTINCT funder_id) funders,
  SUM(CASE WHEN display_name IS NULL OR LENGTH(TRIM(display_name))=0 THEN 1 ELSE 0 END) blank,
  COUNT(amount) has_amount, COUNT(lead_investigator.given_name) has_person, COUNT(lead_investigator.affiliation.name) has_inst,
  COUNT(start_year) has_year, ROUND(SUM(amount)/1000000,1) total_m, MIN(start_year) min_yr, MAX(start_year) max_yr
FROM openalex.awards.nafosted_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) cnt, ROUND(SUM(amount)/1000000,1) total_m
FROM openalex.awards.nafosted_awards GROUP BY 1 ORDER BY 2 DESC LIMIT 12;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='nafosted' AND priority=220;

## §6.5 Net-new go/no-go (post-fold dedup preview)

In [ ]:
%sql
-- §6.5 net-new go/no-go: how many rows survive the fold's lowest-priority-wins dedup.
-- Join the DISTINCT lower-priority id set: crossref sits at BOTH priority 1 and 2, so a
-- plain join fans out and inflates source_rows/shadowed (net_new stays correct).
-- Read: low pct_net_new => mostly already crossref-covered; question whether worth shipping.
SELECT
  COUNT(*) AS source_rows,
  COUNT_IF(x.id IS NULL) AS net_new_rows,
  COUNT_IF(x.id IS NOT NULL) AS shadowed_by_higher_priority,
  ROUND(try_divide(COUNT_IF(x.id IS NULL), COUNT(*)) * 100, 1) AS pct_net_new
FROM openalex.awards.nafosted_awards a
LEFT JOIN (SELECT DISTINCT id FROM openalex.awards.openalex_awards_raw WHERE priority < 220) x
  ON x.id = a.id;